# Hierarchical Lyric Embeddings with BGE-M3

This notebook builds two complementary semantic representations for every eligible song:

1. one normalized embedding for each token-bounded group of lyric lines;
2. one normalized song embedding obtained from the token-count-weighted mean of its group embeddings.

Both levels are preserved. Processing is GPU-accelerated, deterministic, resumable, and written to scope-specific files. The default configuration runs a reproducible 1% pilot; set `run_full_corpus = True` only after validating that pilot.

## 1. Imports and Run Configuration

BGE-M3 provides 1,024-dimensional multilingual dense embeddings and supports long contexts. Groups are deliberately limited to 1,024 model tokens: this keeps batches efficient, preserves local lyric structure, and avoids treating an entire long song as one undifferentiated sequence.

In [9]:
# Keep the embedding stack compatible with the project's PyTorch 2.4 runtime.
import importlib.metadata
import importlib.util
import subprocess
import sys

required_versions = {
    'sentence-transformers': '3.3.1',
    'transformers': '4.46.3',
}
installed_versions = {
    package: importlib.metadata.version(package)
    if importlib.util.find_spec(package.replace('-', '_'))
    else None
    for package in required_versions
}
packages_to_install = [
    f'{package}=={version}'
    for package, version in required_versions.items()
    if installed_versions[package] != version
]
if packages_to_install:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        *packages_to_install,
    ])
    print('Dependencies updated. Restart the kernel before continuing.')
else:
    print(f'Embedding dependencies ready: {installed_versions}')

Embedding dependencies ready: {'sentence-transformers': '3.3.1', 'transformers': '4.46.3'}


In [11]:
import csv
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

model_name = 'BAAI/bge-m3'
gpu_index = 0
run_full_corpus = True
pilot_fraction = 0.01
random_state = 42
min_words_per_document = 10
max_words_per_document = 4000
max_sequence_tokens = 1024
max_content_tokens = 1000
embedding_batch_size = 32
documents_per_shard = 128

sample_fraction = 1.0 if run_full_corpus else pilot_fraction
sample_label = 'full_10to4000w' if run_full_corpus else '1pct_10to4000w'
model_slug = 'bge_m3'
run_label = f'{model_slug}_{sample_label}'

corpus_path = Path('corpus.csv')
export_dir = Path('export')
parts_dir = export_dir / f'.{run_label}_parts'
export_dir.mkdir(exist_ok=True)
parts_dir.mkdir(exist_ok=True)

print(f'Run label: {run_label}')
print(f'Model: {model_name}')
print(f'Maximum sequence length: {max_sequence_tokens:,} tokens')
print(f'Documents per resumable shard: {documents_per_shard:,}')

Run label: bge_m3_full_10to4000w
Model: BAAI/bge-m3
Maximum sequence length: 1,024 tokens
Documents per resumable shard: 128


## 2. Load and Select Eligible Lyrics

The selection rule matches the morphosyntactic notebook: lyrics must contain 10 to 4,000 whitespace-delimited words inclusive. Stable source indices and descriptive metadata are retained at both embedding levels.

In [12]:
try:
    corpus_df = pd.read_csv(corpus_path, encoding='utf-8', on_bad_lines='skip')
except UnicodeDecodeError:
    corpus_df = pd.read_csv(corpus_path, encoding='latin-1', on_bad_lines='skip')

text_column = 'lyrics'
required_columns = {text_column, 'artist', 'title', 'year', 'album', 'url'}
missing_columns = required_columns.difference(corpus_df.columns)
if missing_columns:
    raise KeyError(f'Missing required columns: {sorted(missing_columns)}')

corpus_df = corpus_df.reset_index(names='source_row_index')
text_values = corpus_df[text_column].fillna('').astype(str).str.strip()
word_counts = text_values.str.split().str.len()
eligible_mask = (
    text_values.ne('')
    & word_counts.between(
        min_words_per_document,
        max_words_per_document,
        inclusive='both',
    )
)
eligible_df = corpus_df.loc[eligible_mask].copy()
eligible_df['word_count'] = word_counts.loc[eligible_mask]

if run_full_corpus:
    selected_df = eligible_df.copy()
else:
    selected_df = eligible_df.sample(
        frac=sample_fraction,
        random_state=random_state,
    ).sort_values('source_row_index').copy()

selected_df = selected_df.reset_index(drop=True)
selected_df.insert(0, 'document_id', np.arange(1, len(selected_df) + 1))
assert selected_df['word_count'].between(10, 4000, inclusive='both').all()
if run_full_corpus:
    assert len(selected_df) == len(eligible_df)

metadata_columns = [
    'document_id', 'source_row_index', 'artist', 'title',
    'year', 'album', 'url', 'word_count',
]

print(f'Corpus rows: {len(corpus_df):,}')
print(f'Eligible songs: {len(eligible_df):,}')
print(f'Selected songs: {len(selected_df):,}')
print(f'Observed word range: {selected_df.word_count.min():,}-{selected_df.word_count.max():,}')
selected_df[metadata_columns].head()

Corpus rows: 85,023
Eligible songs: 84,155
Selected songs: 84,155
Observed word range: 10-3,964


,document_id,source_row_index,artist,title,year,album,url,word_count
0,1,0,Dry,94310,2012.0,Tôt ou tard,https://genius.com/Dry-94310-lyrics,673
1,2,1,Mafia K’1 Fry,Au bon vieux temps,2007.0,Jusqu’à la mort,https://genius.com/Mafia-k1-fry-au-bon-vieux-t...,837
2,3,2,DJ Hamida,Attrape-Moi Si Tu Peux,NaN,Mix Party 2015,https://genius.com/Dj-hamida-attrape-moi-si-tu...,306
3,4,3,Kery James,94 c’est le Barça Remix,NaN,NaN,https://genius.com/Kery-james-94-cest-le-barca...,481
4,5,4,Dry,14 ans déjà,2013.0,Maintenant ou jamais,https://genius.com/Dry-14-ans-deja-lyrics,576


## 3. Load BGE-M3 on GPU 0

The model placement is verified from a real parameter rather than inferred from CUDA visibility. Group embeddings are normalized by the model before they are written to disk.

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable; this notebook requires a GPU.')

torch.cuda.set_device(gpu_index)
device = f'cuda:{gpu_index}'
gpu_name = torch.cuda.get_device_name(gpu_index)

embedding_model = SentenceTransformer(model_name, device=device)
embedding_model.max_seq_length = max_sequence_tokens
tokenizer = embedding_model.tokenizer
embedding_dimension = embedding_model.get_sentence_embedding_dimension()
model_device = next(embedding_model.parameters()).device

if model_device.type != 'cuda' or model_device.index != gpu_index:
    raise RuntimeError(f'Model is on {model_device}, expected cuda:{gpu_index}')
if embedding_dimension != 1024:
    raise ValueError(f'Unexpected BGE-M3 dimension: {embedding_dimension}')

print(f'GPU: cuda:{gpu_index} - {gpu_name}')
print(f'Model parameter device: {model_device}')
print(f'Embedding dimension: {embedding_dimension:,}')
print(f'Model sequence limit: {embedding_model.max_seq_length:,}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

GPU: cuda:0 - NVIDIA H100 NVL
Model parameter device: cuda:0
Embedding dimension: 1,024
Model sequence limit: 1,024


## 4. Build Token-Bounded Groups of Lyric Lines

Groups preserve non-empty lyric lines and their original line numbers. Lines are accumulated until adding another line would exceed the content-token budget. Only a single line that exceeds this budget may be split internally.

In [4]:
def build_lyric_groups(raw_text, token_limit=max_content_tokens):
    lines = [
        (line_number, line.strip())
        for line_number, line in enumerate(
            str(raw_text).replace('\r\n', '\n').replace('\r', '\n').split('\n'),
            start=1,
        )
        if line.strip()
    ]
    groups = []
    current_lines = []
    current_token_count = 0

    def flush_current():
        nonlocal current_lines, current_token_count
        if not current_lines:
            return
        groups.append({
            'line_start': current_lines[0][0],
            'line_end': current_lines[-1][0],
            'token_count': current_token_count,
            'group_text': '\n'.join(text for _, text in current_lines),
        })
        current_lines = []
        current_token_count = 0

    for line_number, line_text in lines:
        line_tokens = tokenizer.encode(line_text, add_special_tokens=False)
        if len(line_tokens) > token_limit:
            flush_current()
            for token_start in range(0, len(line_tokens), token_limit):
                token_slice = line_tokens[token_start:token_start + token_limit]
                groups.append({
                    'line_start': line_number,
                    'line_end': line_number,
                    'token_count': len(token_slice),
                    'group_text': tokenizer.decode(
                        token_slice,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    ).strip(),
                })
            continue

        if current_lines and current_token_count + len(line_tokens) > token_limit:
            flush_current()
        current_lines.append((line_number, line_text))
        current_token_count += len(line_tokens)

    flush_current()
    if not groups:
        raise ValueError('An eligible song produced no lyric groups')
    return groups

preview_row = selected_df.iloc[0]
preview_groups = build_lyric_groups(preview_row[text_column])
preview_df = pd.DataFrame(preview_groups)
assert preview_df['token_count'].between(1, max_content_tokens).all()
print(f"Preview song: {preview_row['artist']} - {preview_row['title']}")
print(f'Groups: {len(preview_df):,}')
preview_df[['line_start', 'line_end', 'token_count', 'group_text']].head()

Preview song: ​ihatemed - Ange Déchu
Groups: 1


,line_start,line_end,token_count,group_text
0,1,69,626,Bitch tu m'as déçu\nCombien de temps il faut p...


## 5. Encode Groups and Compute Song Embeddings

Each group is encoded as a normalized `float32` vector. For a song with group vectors $e_i$ and token counts $n_i$, the complete-song representation is

$$
e_{song} = \operatorname{normalize}\left(\frac{\sum_i n_i e_i}{\sum_i n_i}\right).
$$

Every shard is written atomically. Its JSON completion marker is created last, so interrupted shards are recomputed while completed shards are reused.

In [13]:
def save_npy_atomic(path, array):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    with temporary_path.open('wb') as handle:
        np.save(handle, np.asarray(array, dtype=np.float32), allow_pickle=False)
    temporary_path.replace(path)


def save_csv_atomic(path, dataframe):
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    dataframe.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def shard_paths(shard_index):
    prefix = parts_dir / f'part_{shard_index:05d}'
    return {
        'group_embeddings': prefix.with_name(prefix.name + '_group_embeddings.npy'),
        'group_metadata': prefix.with_name(prefix.name + '_group_metadata.csv'),
        'song_embeddings': prefix.with_name(prefix.name + '_song_embeddings.npy'),
        'song_metadata': prefix.with_name(prefix.name + '_song_metadata.csv'),
        'completion': prefix.with_suffix('.json'),
    }


def shard_is_complete(paths):
    return all(path.exists() for path in paths.values())


def remove_partial_shard(paths):
    for path in paths.values():
        path.unlink(missing_ok=True)
        path.with_suffix(path.suffix + '.tmp').unlink(missing_ok=True)


total_shards = math.ceil(len(selected_df) / documents_per_shard)
completed_documents = 0
completed_groups = 0
embedding_seconds = 0.0
peak_gpu_memory_gib = 0.0

for shard_index in range(total_shards):
    paths = shard_paths(shard_index)
    if shard_is_complete(paths):
        with paths['completion'].open('r', encoding='utf-8') as handle:
            shard_summary = json.load(handle)
        completed_documents += shard_summary['documents']
        completed_groups += shard_summary['groups']
        embedding_seconds += shard_summary['embedding_seconds']
        peak_gpu_memory_gib = max(
            peak_gpu_memory_gib,
            shard_summary['peak_gpu_memory_gib'],
        )
    elif any(path.exists() for path in paths.values()):
        remove_partial_shard(paths)


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(gpu_index)

with tqdm(
    total=len(selected_df),
    initial=completed_documents,
    desc='Embedding songs',
    unit='songs',
    dynamic_ncols=True,
) as progress_bar:
    for shard_index, document_start in enumerate(
        range(0, len(selected_df), documents_per_shard)
    ):
        paths = shard_paths(shard_index)
        if shard_is_complete(paths):
            continue

        document_end = min(document_start + documents_per_shard, len(selected_df))
        shard_documents = selected_df.iloc[document_start:document_end]
        group_records = []
        groups_per_document = []

        for row in shard_documents.itertuples(index=False):
            lyric_groups = build_lyric_groups(getattr(row, text_column))
            groups_per_document.append(len(lyric_groups))
            base_metadata = {
                column: getattr(row, column)
                for column in metadata_columns
            }
            for group_index, lyric_group in enumerate(lyric_groups, start=1):
                group_records.append({
                    **base_metadata,
                    'group_id': f'{row.document_id}:{group_index}',
                    'group_index': group_index,
                    **lyric_group,
                })

        group_metadata_df = pd.DataFrame(group_records)
        group_texts = group_metadata_df['group_text'].tolist()

        torch.cuda.synchronize(gpu_index)
        embedding_started = time.perf_counter()
        group_embeddings = embedding_model.encode(
            group_texts,
            batch_size=embedding_batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32, copy=False)
        torch.cuda.synchronize(gpu_index)
        shard_embedding_seconds = time.perf_counter() - embedding_started

        if group_embeddings.shape != (len(group_metadata_df), embedding_dimension):
            raise ValueError(f'Unexpected group embedding shape: {group_embeddings.shape}')
        if not np.isfinite(group_embeddings).all():
            raise ValueError('Non-finite group embeddings detected')

        song_embeddings = []
        song_metadata_records = []
        group_offset = 0
        for row, group_count in zip(
            shard_documents.itertuples(index=False),
            groups_per_document,
        ):
            group_slice = slice(group_offset, group_offset + group_count)
            weights = group_metadata_df.iloc[group_slice]['token_count'].to_numpy(np.float32)
            weighted_embedding = np.average(
                group_embeddings[group_slice],
                axis=0,
                weights=weights,
            )
            weighted_embedding /= np.linalg.norm(weighted_embedding)
            song_embeddings.append(weighted_embedding.astype(np.float32, copy=False))
            song_metadata_records.append({
                **{column: getattr(row, column) for column in metadata_columns},
                'group_count': group_count,
                'group_token_count': int(weights.sum()),
            })
            group_offset += group_count

        song_embeddings = np.vstack(song_embeddings).astype(np.float32, copy=False)
        song_metadata_df = pd.DataFrame(song_metadata_records)
        assert group_offset == len(group_metadata_df)
        assert song_embeddings.shape == (len(shard_documents), embedding_dimension)
        if not np.isfinite(song_embeddings).all():
            raise ValueError('Non-finite song embeddings detected')

        save_npy_atomic(paths['group_embeddings'], group_embeddings)
        save_csv_atomic(paths['group_metadata'], group_metadata_df)
        save_npy_atomic(paths['song_embeddings'], song_embeddings)
        save_csv_atomic(paths['song_metadata'], song_metadata_df)

        current_peak_gib = torch.cuda.max_memory_allocated(gpu_index) / (1024 ** 3)
        shard_summary = {
            'shard_index': shard_index,
            'document_start': document_start,
            'document_end': document_end,
            'documents': len(shard_documents),
            'groups': len(group_metadata_df),
            'embedding_seconds': shard_embedding_seconds,
            'peak_gpu_memory_gib': current_peak_gib,
        }
        completion_temp = paths['completion'].with_suffix('.json.tmp')
        with completion_temp.open('w', encoding='utf-8') as handle:
            json.dump(shard_summary, handle, indent=2)
        completion_temp.replace(paths['completion'])

        completed_documents += len(shard_documents)
        completed_groups += len(group_metadata_df)
        embedding_seconds += shard_embedding_seconds
        peak_gpu_memory_gib = max(peak_gpu_memory_gib, current_peak_gib)
        progress_bar.update(len(shard_documents))
        progress_bar.set_postfix(
            groups=f'{completed_groups:,}',
            groups_s=f'{completed_groups / embedding_seconds:,.1f}',
        )

assert completed_documents == len(selected_df)
print(f'Documents embedded: {completed_documents:,}')
print(f'Groups embedded: {completed_groups:,}')
print(f'Embedding time: {embedding_seconds / 60:.2f} min')
print(f'Throughput: {completed_groups / embedding_seconds:,.2f} groups/s')
print(f'Peak allocated GPU memory: {peak_gpu_memory_gib:.2f} GiB')

Embedding songs:   0%|          | 0/84155 [00:00<?, ?songs/s]

Documents embedded: 84,155
Groups embedded: 104,225
Embedding time: 29.71 min
Throughput: 58.46 groups/s
Peak allocated GPU memory: 3.74 GiB


## 6. Merge Shards into Final Arrays and Metadata

Completed shards are merged in deterministic document order. The final group and song matrices are written as memory-mapped NumPy arrays, while metadata is streamed to CSV. A JSON manifest records the model, pooling rule, dimensions, ordering, and output paths.

In [14]:
shard_summaries = []
for shard_index in range(total_shards):
    paths = shard_paths(shard_index)
    if not shard_is_complete(paths):
        raise FileNotFoundError(f'Incomplete shard: {shard_index}')
    with paths['completion'].open('r', encoding='utf-8') as handle:
        shard_summaries.append(json.load(handle))

total_groups = sum(summary['groups'] for summary in shard_summaries)
total_documents = sum(summary['documents'] for summary in shard_summaries)
assert total_documents == len(selected_df)

group_embeddings_path = export_dir / f'{run_label}_group_embeddings.npy'
group_metadata_path = export_dir / f'{run_label}_group_metadata.csv'
song_embeddings_path = export_dir / f'{run_label}_song_embeddings.npy'
song_metadata_path = export_dir / f'{run_label}_song_metadata.csv'
performance_path = export_dir / f'{run_label}_performance.csv'
manifest_path = export_dir / f'{run_label}_manifest.json'

group_embeddings_temp = group_embeddings_path.with_suffix('.npy.tmp')
song_embeddings_temp = song_embeddings_path.with_suffix('.npy.tmp')
group_metadata_temp = group_metadata_path.with_suffix('.csv.tmp')
song_metadata_temp = song_metadata_path.with_suffix('.csv.tmp')

group_embeddings_final = np.lib.format.open_memmap(
    group_embeddings_temp,
    mode='w+',
    dtype=np.float32,
    shape=(total_groups, embedding_dimension),
)
song_embeddings_final = np.lib.format.open_memmap(
    song_embeddings_temp,
    mode='w+',
    dtype=np.float32,
    shape=(total_documents, embedding_dimension),
)

group_offset = 0
song_offset = 0
for shard_index in tqdm(
    range(total_shards),
    desc='Merging embedding shards',
    unit='shards',
    dynamic_ncols=True,
):
    paths = shard_paths(shard_index)
    shard_group_embeddings = np.load(paths['group_embeddings'], mmap_mode='r')
    shard_song_embeddings = np.load(paths['song_embeddings'], mmap_mode='r')

    next_group_offset = group_offset + len(shard_group_embeddings)
    next_song_offset = song_offset + len(shard_song_embeddings)
    group_embeddings_final[group_offset:next_group_offset] = shard_group_embeddings
    song_embeddings_final[song_offset:next_song_offset] = shard_song_embeddings

    group_metadata_shard = pd.read_csv(paths['group_metadata'])
    song_metadata_shard = pd.read_csv(paths['song_metadata'])
    group_metadata_shard.to_csv(
        group_metadata_temp,
        mode='a' if shard_index else 'w',
        header=shard_index == 0,
        index=False,
    )
    song_metadata_shard.to_csv(
        song_metadata_temp,
        mode='a' if shard_index else 'w',
        header=shard_index == 0,
        index=False,
    )
    group_offset = next_group_offset
    song_offset = next_song_offset

assert group_offset == total_groups
assert song_offset == total_documents
group_embeddings_final.flush()
song_embeddings_final.flush()
del group_embeddings_final, song_embeddings_final

group_embeddings_temp.replace(group_embeddings_path)
song_embeddings_temp.replace(song_embeddings_path)
group_metadata_temp.replace(group_metadata_path)
song_metadata_temp.replace(song_metadata_path)

performance_metrics = pd.DataFrame([{
    'model_name': model_name,
    'gpu_index': gpu_index,
    'gpu_name': gpu_name,
    'run_label': run_label,
    'documents': total_documents,
    'groups': total_groups,
    'embedding_dimension': embedding_dimension,
    'max_sequence_tokens': max_sequence_tokens,
    'embedding_batch_size': embedding_batch_size,
    'embedding_seconds': embedding_seconds,
    'documents_per_second': total_documents / embedding_seconds,
    'groups_per_second': total_groups / embedding_seconds,
    'peak_gpu_memory_gib': peak_gpu_memory_gib,
}])
performance_metrics.to_csv(performance_path, index=False)

manifest = {
    'model_name': model_name,
    'run_label': run_label,
    'selection': {
        'minimum_words': min_words_per_document,
        'maximum_words': max_words_per_document,
        'sample_fraction': sample_fraction,
        'random_state': random_state,
    },
    'grouping': {
        'unit': 'non-empty lyric lines',
        'maximum_content_tokens': max_content_tokens,
        'maximum_model_tokens': max_sequence_tokens,
        'long_line_policy': 'split only when one line exceeds the content-token limit',
    },
    'embeddings': {
        'dtype': 'float32',
        'dimension': embedding_dimension,
        'group_normalization': 'L2',
        'song_pooling': 'token-count-weighted mean of normalized group embeddings',
        'song_normalization': 'L2 after pooling',
        'ordering': 'document_id ascending, then group_index ascending',
    },
    'counts': {
        'documents': total_documents,
        'groups': total_groups,
    },
    'files': {
        'group_embeddings': str(group_embeddings_path),
        'group_metadata': str(group_metadata_path),
        'song_embeddings': str(song_embeddings_path),
        'song_metadata': str(song_metadata_path),
        'performance': str(performance_path),
    },
}
manifest_temp = manifest_path.with_suffix('.json.tmp')
with manifest_temp.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)
manifest_temp.replace(manifest_path)

print(f'Group embeddings: {group_embeddings_path} {total_groups:,} x {embedding_dimension:,}')
print(f'Song embeddings: {song_embeddings_path} {total_documents:,} x {embedding_dimension:,}')
print(f'Manifest: {manifest_path}')
performance_metrics

Merging embedding shards:   0%|          | 0/658 [00:00<?, ?shards/s]

Group embeddings: export/bge_m3_full_10to4000w_group_embeddings.npy 104,225 x 1,024
Song embeddings: export/bge_m3_full_10to4000w_song_embeddings.npy 84,155 x 1,024
Manifest: export/bge_m3_full_10to4000w_manifest.json


,model_name,gpu_index,gpu_name,run_label,documents,groups,embedding_dimension,max_sequence_tokens,embedding_batch_size,embedding_seconds,documents_per_second,groups_per_second,peak_gpu_memory_gib
0,BAAI/bge-m3,0,NVIDIA H100 NVL,bge_m3_full_10to4000w,84155,104225,1024,1024,32,1782.806891,47.203654,58.461183,3.739112


### 6.1 Export Complete Embeddings as CSV

Two wide CSV files are written in aligned row order: one for lyric groups and one for complete songs. Each file includes its metadata columns followed by `embedding_0000` through `embedding_1023`.

CSV is intentionally written in chunks to keep memory bounded. The `.npy` matrices remain the preferred efficient format for numerical work; the CSV files support inspection and interoperability.

In [15]:
# Stream metadata and float32 vectors into interoperable wide CSV files.
csv_chunk_size = 2_000
embedding_column_names = [
    f'embedding_{dimension_index:04d}'
    for dimension_index in range(embedding_dimension)
]


def export_embeddings_to_csv(embeddings_path, metadata_path, output_path):
    embeddings = np.load(embeddings_path, mmap_mode='r')
    temporary_path = output_path.with_suffix('.csv.tmp')
    rows_written = 0

    with temporary_path.open('w', encoding='utf-8', newline='') as output_file:
        writer = csv.writer(output_file)
        metadata_iterator = pd.read_csv(metadata_path, chunksize=csv_chunk_size)
        for metadata_chunk in tqdm(
            metadata_iterator,
            total=math.ceil(len(embeddings) / csv_chunk_size),
            desc=f'Writing {output_path.stem}',
            unit='chunks',
            dynamic_ncols=True,
        ):
            row_end = rows_written + len(metadata_chunk)
            embedding_chunk = np.asarray(embeddings[rows_written:row_end])
            if embedding_chunk.shape != (len(metadata_chunk), embedding_dimension):
                raise ValueError(
                    f'Embedding and metadata alignment failed at row {rows_written:,}'
                )

            if rows_written == 0:
                writer.writerow([*metadata_chunk.columns, *embedding_column_names])

            for metadata_values, embedding_values in zip(
                metadata_chunk.itertuples(index=False, name=None),
                embedding_chunk,
            ):
                writer.writerow([
                    *metadata_values,
                    *(f'{value:.8g}' for value in embedding_values),
                ])
            rows_written = row_end

    if rows_written != len(embeddings):
        raise ValueError(f'Expected {len(embeddings):,} rows, wrote {rows_written:,}')
    temporary_path.replace(output_path)
    return rows_written


group_embeddings_csv_path = export_dir / f'{run_label}_group_embeddings.csv'
song_embeddings_csv_path = export_dir / f'{run_label}_song_embeddings.csv'

group_csv_rows = export_embeddings_to_csv(
    group_embeddings_path,
    group_metadata_path,
    group_embeddings_csv_path,
)
song_csv_rows = export_embeddings_to_csv(
    song_embeddings_path,
    song_metadata_path,
    song_embeddings_csv_path,
)

assert group_csv_rows == total_groups
assert song_csv_rows == total_documents
print(f'Group embedding CSV: {group_embeddings_csv_path} ({group_csv_rows:,} rows)')
print(f'Song embedding CSV: {song_embeddings_csv_path} ({song_csv_rows:,} rows)')

Writing bge_m3_full_10to4000w_group_embeddings:   0%|          | 0/53 [00:00<?, ?chunks/s]

Writing bge_m3_full_10to4000w_song_embeddings:   0%|          | 0/43 [00:00<?, ?chunks/s]

Group embedding CSV: export/bge_m3_full_10to4000w_group_embeddings.csv (104,225 rows)
Song embedding CSV: export/bge_m3_full_10to4000w_song_embeddings.csv (84,155 rows)


## 7. Reload and Validate Both Embedding Levels

Validation checks array dimensions, `float32` storage, finite values, near-unit norms, metadata alignment, and group-to-song cosine similarity. A small nearest-neighbor inspection provides a semantic sanity check without constructing a full pairwise similarity matrix.

In [8]:
group_embeddings_check = np.load(group_embeddings_path, mmap_mode='r')
song_embeddings_check = np.load(song_embeddings_path, mmap_mode='r')
song_metadata_check = pd.read_csv(song_metadata_path)

assert group_embeddings_check.shape == (total_groups, embedding_dimension)
assert song_embeddings_check.shape == (total_documents, embedding_dimension)
assert group_embeddings_check.dtype == np.float32
assert song_embeddings_check.dtype == np.float32
assert len(song_metadata_check) == total_documents
assert song_metadata_check['document_id'].is_monotonic_increasing


def validate_embedding_matrix(matrix, chunk_size=100_000):
    minimum_norm = float('inf')
    maximum_norm = 0.0
    for start in tqdm(
        range(0, len(matrix), chunk_size),
        desc='Validating embedding norms',
        unit='chunks',
        dynamic_ncols=True,
    ):
        chunk = np.asarray(matrix[start:start + chunk_size])
        if not np.isfinite(chunk).all():
            raise ValueError('Non-finite embedding value detected')
        norms = np.linalg.norm(chunk, axis=1)
        minimum_norm = min(minimum_norm, float(norms.min()))
        maximum_norm = max(maximum_norm, float(norms.max()))
    if minimum_norm < 0.999 or maximum_norm > 1.001:
        raise ValueError(f'Unexpected norm range: {minimum_norm:.6f}-{maximum_norm:.6f}')
    return minimum_norm, maximum_norm


group_norm_range = validate_embedding_matrix(group_embeddings_check)
song_norm_range = validate_embedding_matrix(song_embeddings_check)

group_metadata_rows = 0
for metadata_chunk in pd.read_csv(group_metadata_path, chunksize=100_000):
    group_metadata_rows += len(metadata_chunk)
assert group_metadata_rows == total_groups

query_positions = np.linspace(
    0,
    total_documents - 1,
    num=min(5, total_documents),
    dtype=int,
)
neighbor_rows = []
for query_position in query_positions:
    query_embedding = np.asarray(song_embeddings_check[query_position])
    similarities = np.asarray(song_embeddings_check @ query_embedding)
    similarities[query_position] = -np.inf
    neighbor_positions = np.argpartition(similarities, -3)[-3:]
    neighbor_positions = neighbor_positions[np.argsort(similarities[neighbor_positions])[::-1]]
    query_metadata = song_metadata_check.iloc[query_position]
    for neighbor_rank, neighbor_position in enumerate(neighbor_positions, start=1):
        neighbor_metadata = song_metadata_check.iloc[neighbor_position]
        neighbor_rows.append({
            'query_document_id': int(query_metadata['document_id']),
            'query': f"{query_metadata['artist']} - {query_metadata['title']}",
            'rank': neighbor_rank,
            'neighbor_document_id': int(neighbor_metadata['document_id']),
            'neighbor': f"{neighbor_metadata['artist']} - {neighbor_metadata['title']}",
            'cosine_similarity': float(similarities[neighbor_position]),
        })
nearest_neighbors_df = pd.DataFrame(neighbor_rows)
nearest_neighbors_path = export_dir / f'{run_label}_nearest_neighbors_check.csv'
nearest_neighbors_df.to_csv(nearest_neighbors_path, index=False)

comparison_song_position = int(song_metadata_check['group_count'].to_numpy().argmax())
comparison_song = song_metadata_check.iloc[comparison_song_position]
comparison_document_id = int(comparison_song['document_id'])
comparison_group_count = int(comparison_song['group_count'])
comparison_group_start = int(
    song_metadata_check.iloc[:comparison_song_position]['group_count'].sum()
)
comparison_group_end = comparison_group_start + comparison_group_count

comparison_metadata_chunks = []
for metadata_chunk in pd.read_csv(group_metadata_path, chunksize=100_000):
    matching_rows = metadata_chunk[
        metadata_chunk['document_id'].eq(comparison_document_id)
    ]
    if not matching_rows.empty:
        comparison_metadata_chunks.append(matching_rows)
comparison_group_metadata = pd.concat(comparison_metadata_chunks, ignore_index=True)
assert len(comparison_group_metadata) == comparison_group_count

comparison_group_embeddings = np.asarray(
    group_embeddings_check[comparison_group_start:comparison_group_end]
)
comparison_similarities = comparison_group_embeddings @ np.asarray(
    song_embeddings_check[comparison_song_position]
)
comparison_group_metadata['cosine_to_song'] = comparison_similarities
comparison_group_metadata = comparison_group_metadata.sort_values(
    'cosine_to_song',
    ascending=False,
)
group_similarity_path = export_dir / f'{run_label}_group_to_song_similarity_check.csv'
comparison_group_metadata.to_csv(group_similarity_path, index=False)

print(f'Group norm range: {group_norm_range[0]:.6f}-{group_norm_range[1]:.6f}')
print(f'Song norm range: {song_norm_range[0]:.6f}-{song_norm_range[1]:.6f}')
print(f'Validated group metadata rows: {group_metadata_rows:,}')
print('\nNearest songs:')
display(nearest_neighbors_df)
print(
    f"\nGroups closest to {comparison_song['artist']} - "
    f"{comparison_song['title']} ({comparison_group_count} groups):"
)
display(comparison_group_metadata.head(5)[
    ['group_id', 'line_start', 'line_end', 'token_count', 'group_text', 'cosine_to_song']
])
print('\nGroups farthest from the same song embedding:')
display(comparison_group_metadata.tail(5).sort_values('cosine_to_song')[
    ['group_id', 'line_start', 'line_end', 'token_count', 'group_text', 'cosine_to_song']
])

Validating embedding norms:   0%|          | 0/1 [00:00<?, ?chunks/s]

Validating embedding norms:   0%|          | 0/1 [00:00<?, ?chunks/s]

Group norm range: 1.000000-1.000000
Song norm range: 1.000000-1.000000
Validated group metadata rows: 1,039

Nearest songs:


,query_document_id,query,rank,neighbor_document_id,neighbor,cosine_similarity
0,1,​ihatemed - Ange Déchu,1,687,Henri Bleu - Jester,0.687832
1,1,​ihatemed - Ange Déchu,2,435,L’Entourage - L’Entourage en live dans Planète...,0.684760
2,1,​ihatemed - Ange Déchu,3,252,Sopico - Unplugged #3: Arbre de vie,0.680824
3,211,Leo Roi - La carotte,1,342,Grünt - Grünt #36,0.606013
4,211,Leo Roi - La carotte,2,435,L’Entourage - L’Entourage en live dans Planète...,0.589239
5,211,Leo Roi - La carotte,3,254,HPA MOB - Princes Guerriers,0.583876
6,421,Les Alchimistes - Russuk,1,315,A2H - Blues,0.683362
7,421,Les Alchimistes - Russuk,2,342,Grünt - Grünt #36,0.672494
8,421,Les Alchimistes - Russuk,3,730,Raplume - Météorite,0.671661
9,631,Still Fresh - NE M’EN VEUX PAS,1,435,L’Entourage - L’Entourage en live dans Planète...,0.638058



Groups closest to L’Entourage - L’Entourage en live dans Planète Rap (8 groups):


,group_id,line_start,line_end,token_count,group_text,cosine_to_song
0,435:1,2,66,987,J'ai la gorge serrée et la peur du futur\nComm...,0.860681
2,435:3,129,198,1000,"Sale tass, tu parles mal, je t'avais dit, y'a ...",0.842684
6,435:7,414,490,994,"Un jour un parigot m'a dit : « Ici, on salit t...",0.837498
3,435:4,199,266,982,"Eff Gee fabuleux, t'as vu le style, fils, j'al...",0.828822
4,435:5,267,336,996,"Chouf, j'te l'avais dit, là, t'es maîtrisé à t...",0.821213



Groups farthest from the same song embedding:


,group_id,line_start,line_end,token_count,group_text,cosine_to_song
7,435:8,491,493,43,Cette vie nous rend sceptique donc on l'ironis...,0.563368
5,435:6,337,413,1000,"Être plus fort qu'les autres, c'est juste un p...",0.805174
1,435:2,67,128,999,J'rêve de baiser une fliquette et que cette ch...,0.812792
4,435:5,267,336,996,"Chouf, j'te l'avais dit, là, t'es maîtrisé à t...",0.821213
3,435:4,199,266,982,"Eff Gee fabuleux, t'as vu le style, fils, j'al...",0.828822


## 8. Pilot Summary and Full-Run Estimate

The report records measured pilot performance and a linear full-corpus estimate. The estimate is advisory: actual duration depends on the complete token-length distribution, batching efficiency, and concurrent GPU load.

In [10]:
full_scale_factor = len(eligible_df) / len(selected_df)
estimated_full_groups = round(total_groups * full_scale_factor)
estimated_full_minutes = estimated_full_groups / (
    total_groups / embedding_seconds
) / 60
estimated_group_embedding_gib = (
    estimated_full_groups * embedding_dimension * np.dtype(np.float32).itemsize
    / (1024 ** 3)
)
estimated_song_embedding_gib = (
    len(eligible_df) * embedding_dimension * np.dtype(np.float32).itemsize
    / (1024 ** 3)
)

with manifest_path.open('r', encoding='utf-8') as handle:
    manifest = json.load(handle)
manifest['files']['nearest_neighbors_check'] = str(nearest_neighbors_path)
manifest['files']['group_to_song_similarity_check'] = str(group_similarity_path)
manifest['full_run_estimate'] = {
    'groups': estimated_full_groups,
    'minutes': estimated_full_minutes,
    'group_embedding_gib': estimated_group_embedding_gib,
    'song_embedding_gib': estimated_song_embedding_gib,
    'method': 'linear extrapolation from the measured pilot',
}
manifest_temp = manifest_path.with_suffix('.json.tmp')
with manifest_temp.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)
manifest_temp.replace(manifest_path)

summary_report = f"""
BGE-M3 HIERARCHICAL LYRIC EMBEDDING REPORT
==========================================
Run label: {run_label}
Model: {model_name}
GPU: cuda:{gpu_index} - {gpu_name}

Measured run
------------
Songs: {total_documents:,}
Lyric groups: {total_groups:,}
Embedding dimension: {embedding_dimension:,}
Embedding time: {embedding_seconds:.2f} seconds
Throughput: {total_groups / embedding_seconds:.2f} groups/s
Peak allocated GPU memory: {peak_gpu_memory_gib:.2f} GiB

Representations retained
------------------------
Group level: one normalized embedding per token-bounded group of lyric lines
Song level: token-count-weighted mean of normalized group vectors, then L2 normalization

Linear full-corpus estimate
---------------------------
Eligible songs: {len(eligible_df):,}
Estimated groups: {estimated_full_groups:,}
Estimated embedding time: {estimated_full_minutes:.1f} minutes
Estimated final group matrix: {estimated_group_embedding_gib:.2f} GiB
Estimated final song matrix: {estimated_song_embedding_gib:.2f} GiB

Validation
----------
Group norm range: {group_norm_range[0]:.6f}-{group_norm_range[1]:.6f}
Song norm range: {song_norm_range[0]:.6f}-{song_norm_range[1]:.6f}
Group metadata rows: {group_metadata_rows:,}
""".strip()

report_path = export_dir / f'{run_label}_report.txt'
report_path.write_text(summary_report + '\n', encoding='utf-8')
print(summary_report)
print(f'\nReport: {report_path}')

BGE-M3 HIERARCHICAL LYRIC EMBEDDING REPORT
Run label: bge_m3_1pct_10to4000w
Model: BAAI/bge-m3
GPU: cuda:0 - NVIDIA H100 NVL

Measured run
------------
Songs: 842
Lyric groups: 1,039
Embedding dimension: 1,024
Embedding time: 17.43 seconds
Throughput: 59.60 groups/s
Peak allocated GPU memory: 3.74 GiB

Representations retained
------------------------
Group level: one normalized embedding per token-bounded group of lyric lines
Song level: token-count-weighted mean of normalized group vectors, then L2 normalization

Linear full-corpus estimate
---------------------------
Eligible songs: 84,155
Estimated groups: 103,844
Estimated embedding time: 29.0 minutes
Estimated final group matrix: 0.40 GiB
Estimated final song matrix: 0.32 GiB

Validation
----------
Group norm range: 1.000000-1.000000
Song norm range: 1.000000-1.000000
Group metadata rows: 1,039

Report: export/bge_m3_1pct_10to4000w_report.txt


## 9. Retain Final CSV Deliverables Only

Once CSV export and validation have completed, the standalone metadata CSVs are redundant because their columns are already embedded in the two wide CSV files. This final cleanup retains only the complete group-level and song-level CSV deliverables, reports, manifests, and lightweight diagnostic outputs.

In [17]:
# Keep the self-contained wide CSV exports and remove derived duplicates.
with manifest_path.open('r', encoding='utf-8') as handle:
    manifest = json.load(handle)

manifest['files'] = {
    'group_embeddings_csv': str(group_embeddings_csv_path),
    'song_embeddings_csv': str(song_embeddings_csv_path),
    'performance': str(performance_path),
}
manifest['retained_deliverables'] = {
    'embedding_format': 'CSV with metadata columns followed by embedding_0000 to embedding_1023',
    'group_rows': group_csv_rows,
    'song_rows': song_csv_rows,
}
manifest_temp = manifest_path.with_suffix('.json.tmp')
with manifest_temp.open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)
manifest_temp.replace(manifest_path)

for intermediate_path in [
    group_metadata_path,
    song_metadata_path,
    group_embeddings_path,
    song_embeddings_path,
]:
    intermediate_path.unlink(missing_ok=True)

if parts_dir.exists():
    for shard_path in parts_dir.glob('*'):
        shard_path.unlink()
    parts_dir.rmdir()

print(f'Retained: {group_embeddings_csv_path}')
print(f'Retained: {song_embeddings_csv_path}')
print('Removed standalone metadata, binary arrays, and resumable shards.')

Retained: export/bge_m3_full_10to4000w_group_embeddings.csv
Retained: export/bge_m3_full_10to4000w_song_embeddings.csv
Removed standalone metadata, binary arrays, and resumable shards.
